# BootstrapFinetune (Apple Silicon / MPS)

Bootstrap successful traces into training data, then update model weights rather than only prompt state.

**Use it when:** You control a trainable model and want to distill accepted DSPy traces into a reusable local adapter.

**What compilation changes:** A PEFT LoRA adapter for Qwen2.5-0.5B-Instruct; the prompt remains separately inspectable.

Important configuration in this benchmark:

- DSPy BootstrapFinetune and native LocalProvider with a balanced-trace validation guard
- thin MacLocalProvider subclass selects Transformers/MPS serving and adapts the pinned TRL argument name
- 10 epochs, batch size 1, gradient accumulation 4, PEFT LoRA, learning rate 2e-4
- GPT-5.6 Sol teacher; Qwen2.5-0.5B student trained locally

Every notebook loads the canonical 300-row expanded dataset and frozen,
pair-grouped membership: 160 training, 60 validation, and 80 locked-test rows.
A semantic human/AI pair can never cross partitions. Optimizer choices use
validation only; the locked test is released once after the program is frozen.
These scores teach optimizer tradeoffs, not a general AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "expanded_notebooks" / "comparison.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'bootstrap-finetune'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='bootstrap-finetune'; live=False
train=160 (human=80, AI=80); validation=60 (human=30, AI=30); test=80 (human=40, AI=40)


## LocalProvider on Apple Silicon and with standard SGLang

DSPy 3.2.1's `LocalProvider` has two separate jobs. Its **training** path
already uses Transformers and TRL and selects CUDA, then MPS, then CPU.
We keep that path: DSPy formats the accepted traces, masks non-assistant
tokens, constructs the PEFT trainer, trains, and saves the merged model.

Its standard **serving** path starts `python -m sglang.launch_server`.
That is the path to use on an SGLang-compatible CUDA system:

```python
from dspy.clients.lm_local import LocalProvider

student_lm = dspy.LM(
    "openai/local:Qwen/Qwen2.5-0.5B-Instruct",
    provider=LocalProvider(),
    cache=False,
    max_tokens=96,
)
```

Apple Silicon needs a different serving backend. This chapter's
`MacLocalProvider(LocalProvider)` overrides only `launch` and `kill` so
the model is loaded by Transformers on MPS. Its `finetune` method calls
`LocalProvider.finetune` unchanged except for translating DSPy 3.2.1's
`max_seq_length` keyword to the `max_length` name used by the pinned TRL
0.24.0. There is no replacement training loop. The complete small shim is
in `chapter06/apple_finetune.py`; the optimizer call below is identical on
both platforms.

## Compile shape

This is the essential DSPy call used by the shared executable runner:

```python
optimizer = BalancedBootstrapFinetune(
    metric=exact_match, train_kwargs=training_config,
    exclude_demos=True, num_threads=1, min_examples_per_class=2,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, teacher=sol_teacher,
)
```

`compile` returns a program. The shared runner then evaluates that program on the
untouched 80-row locked test split. The baseline has its own notebook; all other
notebooks report validation and locked-test accuracy separately.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: bootstrap-finetune
task model: Qwen/Qwen2.5-0.5B-Instruct
final test accuracy: 70.0% (56/80)
optimized validation accuracy: 70.0%
same-model baseline: 51.2%
uplift: +18.8 points
accepted traces: human=77, AI=63
optimization cost: $0.8651
optimization time: 1026.3s
mean inference latency: 1.526s
p95 inference latency: 1.992s

Published artifacts:
- program snapshot: chapter06/results/expanded_notebooks/bootstrap-finetune/full/optimized_program.json
- prompt snapshot: chapter06/results/expanded_notebooks/bootstrap-finetune/full/learned_prompt.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Check the accepted human/AI trace counts before the score. A balanced accepted set is a prerequisite for interpreting this fine-tune as a classification experiment.

The saved output above uses the checked-in expanded-dataset result, so opening or
rerunning the notebook is free. The paid run first passed a bounded smoke profile,
then froze its full program using training and validation only. Set
`CHAPTER06_RUN_LIVE=1` before launching Jupyter to reproduce that full protocol;
prompt optimizers require an OpenAI key, while weight optimizers also require the
local PyTorch/Transformers stack. The next cell previews the durable program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final test accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`; machine-readable
scores, prompts, programs, predictions, timing, cost, versions, hashes, failures,
and retries live under `results/expanded_notebooks/`. Weight-model payloads are
generated locally and Git-ignored; their checked-in manifests retain file hashes,
sizes, configuration, prompts, programs, and scores. Credentials and provider
request bodies are intentionally excluded.